In [ ]:
%load_ext autoreload
%autoreload 2

import modules
from modules.schnorr_lattice import SchnorrAlgQAOA


import numpy as np
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram

from more_itertools import distinct_permutations

from itertools import islice

In [3]:
seed = 99

In [4]:
N5 = 48567227

In [14]:
from math import log2, log, ceil

In [15]:
m = ceil(log2(48567227))
m

26

In [20]:
n = round(1.5*m / ceil(log2(m)))
print(n)
print(type(n))

8
<class 'int'>


In [10]:
cvp = SchnorrAlgQAOA(N5, 3, 1.5, seed)

El numero de bits de N = 48567227 es m = 26
La dimension del reticulo que vamos a tratar es n = 8
La cota smooth que vamos a tomar: 64


In [ ]:
#cvp.set_n(11)
#cvp.set_smoothbound(242)

In [320]:
cvp.set_n(8)
cvp.set_smoothbound(128)

In [321]:
diag = [(i + 1) // 2 for i in range(1, cvp.get_n() + 1)]

In [322]:
pares = set()

In [323]:
max_iter = 130
max_sr = 130
i = 0

In [324]:
for p in distinct_permutations(diag):
    
    if i >= max_iter or len(pares) >= max_sr:
        break
    
    B, t = cvp.generate_cvp(10, False, p)

    D, b_op, res_vector, step_signs, w, dist  = cvp.babai_algorithm(B, t, 0.75)

    qubo = cvp.define_qubo(D, res_vector, step_signs)

    Hc, _ = cvp.define_hamiltonian(qubo)

    circuit = cvp.construct_circuit(Hc, reps = 1)

    x0 = np.asarray([0.0]*circuit.num_parameters)

    optParameters = cvp.qaoa_algorithm(circuit, Hc, x0)

    results = cvp.sample_from_parameters(circuit, optParameters, 100_000)


    resultk = dict(islice(results.items(), 31))

    nD = cvp.integer_to_matrix(D)

    vnews = cvp.bitstring2latticeVectors(nD, resultk.keys(), step_signs, b_op)

    nB = cvp.integer_to_matrix(B)

    uv_pairs = cvp.vectors2uv_pairs(nB, vnews)

    sr_pairs = cvp.uv_pairs2sr_pairs(uv_pairs)
    print(f'Iteracion {i} encontrado {len(sr_pairs)} SR Pairs')

    

    for par in sr_pairs:
        pares.add(par)

    print(pares)

    i = i + 1

Iteracion 0 encontrado 7 SR Pairs
{(48388725, 1), (48550320, 1), (48600552, 1), (48580560, 1), (242712288, 5), (48558510, 1), (823488120, 17)}
Iteracion 1 encontrado 5 SR Pairs
{(48388725, 1), (48550320, 1), (48600552, 1), (48580560, 1), (242712288, 5), (48558510, 1), (48243195, 1), (823488120, 17), (145645885, 3)}
Iteracion 2 encontrado 6 SR Pairs
{(48388725, 1), (48550320, 1), (483810327, 10), (48294792, 1), (48600552, 1), (48580560, 1), (242712288, 5), (48558510, 1), (48243195, 1), (823488120, 17), (145645885, 3)}
Iteracion 3 encontrado 9 SR Pairs
{(48388725, 1), (242295543, 5), (48550320, 1), (483810327, 10), (48294792, 1), (48600552, 1), (48580560, 1), (242712288, 5), (48558510, 1), (48243195, 1), (194260275, 4), (823488120, 17), (145645885, 3)}
Iteracion 4 encontrado 4 SR Pairs
{(48388725, 1), (242295543, 5), (48550320, 1), (483810327, 10), (48294792, 1), (48600552, 1), (48580560, 1), (242712288, 5), (48558510, 1), (48243195, 1), (194260275, 4), (823488120, 17), (145645885, 3)}
I

In [325]:
print(pares)

{(775602135, 16), (193568375, 4), (48973260, 1), (48681325, 1), (48481875, 1), (825384339, 17), (825968000, 17), (146060600, 3), (2484356875, 51), (48534915, 1), (194260275, 4), (193978125, 4), (48620000, 1), (146895232, 3), (145201420, 3), (97435855, 2), (1067895171, 22), (777803125, 16), (1021904000, 21), (194041575, 4), (146132272, 3), (48432384, 1), (48506848, 1), (632187500, 13), (48783735, 1), (48538616, 1), (48600552, 1), (145775000, 3), (146656250, 3), (1556806875, 32), (2118933999, 44), (48550320, 1), (48518400, 1), (145546856, 3), (532390887, 11), (195270075, 4), (631497300, 13), (340711800, 7), (48412000, 1), (48553700, 1), (438077471, 9), (48372480, 1), (290697055, 6), (633080000, 13), (48388725, 1), (436810000, 9), (632491200, 13), (97226325, 2), (828100000, 17), (145959520, 3), (48243195, 1), (242712288, 5), (1208442807, 25), (3059969000, 63), (146410000, 3), (824107284, 17), (48558510, 1), (97060425, 2), (96702579, 2), (145645885, 3), (628645941, 13), (242266739, 5), (14

In [327]:
print(len(pares))

124


In [1]:
import math


In [3]:
N = 624911573291

In [4]:
logN = round(math.log2(N))
logN

39

In [7]:
m = int(3/2*int(logN)//int(round(math.log2(logN))))
m

11